# OpenMLS v5 statistics: RFC 9420 structure and IoT/device performance

This notebook sources `statistics_analysis_openmls_v5.R` and keeps the analysis logic in the reusable R workflow. It reads OpenMLS benchmark outputs under `OpenMLS_containerized/benchmark_output/`, uses operation-local OpenMLS fields where benchmark context is stale or incomplete, and treats current partial/smoke-like data as exploratory.

In [ ]:
getwd()
source("statistics_analysis_openmls_v5.R")

input_dir <- "OpenMLS_containerized/benchmark_output"
out_dir <- "analysis_output/openmls_v5"
file_batch_size <- openmls_v5_default_file_batch_size
chunk_rows <- openmls_v5_default_chunk_rows
use_cache <- FALSE

if (requireNamespace("repr", quietly = TRUE)) {
  options(repr.plot.width = 12, repr.plot.height = 7)
}

show_v5_plot <- function(name) {
  cat("\n## ", name, "\n", sep = "")
  result <- get(name, mode = "function")(openmls_v5_df)
  print_plot_or_skip(result)
  invisible(result)
}

cat("input_dir=", input_dir, "\n", sep = "")
cat("out_dir=", out_dir, "\n", sep = "")
cat("file_batch_size=", file_batch_size, "\n", sep = "")
cat("chunk_rows=", chunk_rows, "\n", sep = "")

## Dataset Discovery And Run Inventory

In [ ]:
openmls_v5_runs <- discover_openmls_runs(input_dir)
openmls_v5_runs

In [ ]:
openmls_v5_raw <- read_openmls_v5_raw(
  input_dir = input_dir,
  use_cache = use_cache,
  cache_dir = file.path(out_dir, "cache"),
  file_batch_size = file_batch_size,
  chunk_rows = chunk_rows,
  keep_all_columns = openmls_v5_default_keep_all_columns
)
openmls_v5_df <- normalize_openmls_v5(openmls_v5_raw)
cat("Rows loaded: ", format(nrow(openmls_v5_df), big.mark = ","), "\n", sep = "")
summarize_runs(openmls_v5_df)

## Schema And Missingness Report

In [ ]:
check_required_metrics(openmls_v5_df) |>
  dplyr::arrange(plot_name, required_field)

important_missingness(openmls_v5_df) |>
  dplyr::filter(percent_missing > 0) |>
  dplyr::arrange(dplyr::desc(percent_missing), operation_family, field) |>
  dplyr::slice_head(n = 40)

## Operation Counts And Device Coverage

In [ ]:
summarize_operations(openmls_v5_df) |>
  dplyr::filter(operation %in% openmls_v5_parent_operations)

summarize_devices(openmls_v5_df)

## A. Dataset And IoT/device Overview

In [ ]:
for (name in c(
  "plot_device_operation_coverage",
  "plot_max_group_size_by_device_operation",
  "plot_device_slowdown_vs_container"
)) show_v5_plot(name)

## B. RFC 9420 Structural Sanity

In [ ]:
for (name in c(
  "plot_update_direct_path_scaling",
  "plot_update_hpke_identity",
  "plot_update_path_nodes_identity",
  "plot_welcome_size_by_recipient_count",
  "plot_ratchet_tree_bytes_by_tree_size",
  "plot_ciphertext_vs_plaintext"
)) show_v5_plot(name)

## C. SelfUpdate / Update

In [ ]:
for (name in c(
  "plot_update_wall_time_loess",
  "plot_update_alloc_loess",
  "plot_update_child_span_decomposition",
  "plot_update_gam_surface"
)) show_v5_plot(name)

## D. Add + Welcome

In [ ]:
for (name in c(
  "plot_add_wall_time_loess",
  "plot_add_welcome_split",
  "plot_welcome_secret_count_identity",
  "plot_welcome_bytes_by_device"
)) show_v5_plot(name)

## E. Remove

In [ ]:
for (name in c(
  "plot_remove_tree_before_after",
  "plot_remove_truncation_duration",
  "plot_remove_truncated_levels_distribution",
  "plot_remove_wall_time"
)) show_v5_plot(name)

## F. JoinFromWelcome

In [ ]:
for (name in c(
  "plot_join_wall_vs_ratchet_tree_bytes",
  "plot_join_payload_components",
  "plot_join_delivery_mode_boxplot",
  "plot_join_gam_surface"
)) show_v5_plot(name)

## G. ApplicationMessageCreate

In [ ]:
for (name in c(
  "plot_app_create_wall_vs_plaintext",
  "plot_app_create_payload_boxplot",
  "plot_app_create_generation_effect",
  "plot_app_create_child_span_comparison",
  "plot_app_create_gam_surface"
)) show_v5_plot(name)

## H. ApplicationMessageReceive

In [ ]:
for (name in c(
  "plot_app_receive_wall_vs_ciphertext",
  "plot_app_receive_generation_gap",
  "plot_app_receive_first_receive",
  "plot_app_receive_child_span_comparison",
  "plot_app_receive_gam_surface"
)) show_v5_plot(name)

## I. CommitReceive

In [ ]:
for (name in c(
  "plot_commit_receive_wall_by_member_count",
  "plot_commit_receive_wall_by_commit_size",
  "plot_commit_receive_child_decomposition",
  "plot_commit_receive_receiver_position",
  "plot_commit_receive_sampling_coverage",
  "plot_commit_receive_parent_child_consistency",
  "plot_commit_receive_gam_surface"
)) show_v5_plot(name)

## J. Resources / IoT

In [ ]:
for (name in c(
  "plot_resource_alloc_by_operation_device",
  "plot_resource_rss_by_operation_device",
  "plot_resource_cpu_by_operation_device",
  "plot_resource_throttling",
  "plot_iot_feasibility_frontier",
  "plot_device_ranking_by_operation"
)) show_v5_plot(name)

## K. Summary Dashboard-style Views

In [ ]:
for (name in c(
  "plot_operation_p95_overview",
  "plot_missingness_heatmap"
)) show_v5_plot(name)

## Plot Files Written

In [ ]:
table_paths <- write_openmls_v5_tables(openmls_v5_df, out_dir)
plot_result <- run_all_openmls_v5_plots(openmls_v5_df, out_dir)
plot_result$created
cat("Plot directory: ", file.path(out_dir, "plots"), "\n", sep = "")
cat("Table directory: ", file.path(out_dir, "tables"), "\n", sep = "")

## Skipped Plots And Reasons

In [ ]:
plot_result$skipped

## Final Caveats And Next Steps

In [ ]:
cat("Caveats:\n")
cat("- Current runs are useful for exploratory OpenMLS operation-level analysis.\n")
cat("- Do not make final thesis claims from partial/smoke-only data or stale benchmark context fields.\n")
cat("- CommitReceive is included and should stay separated by commit_create_op.\n")
cat("- External-device rows are included where present and kept separate from container rows.\n")
cat("- Rerun this notebook after larger benchmark runs; the reader batches files and CSV row chunks.\n")